# Nurse Navigation - Repeat Callers

Measures how often the same person generates another 911 call transferred to nurse navigation within 30, 60, and 90 days.

Logis has no single patient identifier, so each person is resolved from patient first name, last name, date of birth, and phone. Exact matches are applied first, followed by tightly limited fuzzy rules for spelling differences, nicknames, and last-name changes. Every link is tagged with the rule that produced it, so link counts can be reviewed by rule.

Definitions:

- Index call: any call from an identified person.
- Repeat (return): a separate 911 incident from the same person within the window after the index call.
- Return rate: share of index calls followed by a return within the window. Only index calls with a full follow-up window before the end of the data are counted, so calls in the final 30, 60, or 90 days do not lower the rate.

No names, dates of birth, or phone numbers are written to the Excel output. Persons are labeled with a generated person key.

## 1. Setup

In [ ]:
import os, re, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200); pd.set_option("display.max_colwidth", 200)
plt.rcParams.update({"figure.figsize":(11,5),"figure.dpi":110,"axes.grid":True,"grid.alpha":0.25,
    "axes.spines.top":False,"axes.spines.right":False,"font.size":11,"axes.titlesize":13,"axes.titleweight":"bold"})
TEAL, NAVY, CORAL, GOLD, GREY = "#028090","#0B2545","#D1495B","#E0A500","#8FA0A6"

DATA_DIR   = "/Workspace/Users/josh.smitherman@gmr.net/nurse_nav/data"
OUT_DIR    = "/Workspace/Users/josh.smitherman@gmr.net/nurse_nav/results"
PROMPT_DIR = "/Workspace/Users/josh.smitherman@gmr.net/nurse_nav/prompts"
SOURCE_FILE = "data_april2026-aug2026.xlsx"

WINDOWS = [30, 60, 90]
FREQUENT_THRESHOLDS = [2, 3, 5]
NAME_MIN_SIMILARITY = 0.88
MAX_LAST_NAMES_PER_DOB = 25
MAX_FIRST_NAME_VARIANTS = 3
MAX_LAST_NAME_VARIANTS = 3
REVIEW_PHONES_PER_PERSON = 5
MAX_PEOPLE_PER_PHONE = 5
USE_PERSONAL_ID = False
QA_SAMPLE_N = 60
RANDOM_SEED = 42

os.makedirs(OUT_DIR, exist_ok=True)
RUN_ID = pd.Timestamp.now().strftime("%Y%m%d_%H%M")
RESULTS = {}
def keep(d, n): RESULTS[n] = d.copy(); return d
print("run:", RUN_ID)

## 2. Load calls and resolve columns

In [ ]:
def clean_col(c): return re.sub(r"_+","_",re.sub(r"[^\w]+","_",str(c).strip())).lower()
raw = pd.read_excel(os.path.join(DATA_DIR, SOURCE_FILE)); raw.columns = [clean_col(c) for c in raw.columns]
def find_col(df, exact, contains=None):
    norm = lambda x: x.strip("_"); nrm = {norm(c): c for c in df.columns}
    for c in exact:
        if c in df.columns: return c
        if norm(c) in nrm: return nrm[norm(c)]
    for pat in (contains or []):
        hits = [c for c in df.columns if pat in c]
        if hits: return sorted(hits, key=len)[0]
    return None

DATE        = find_col(raw, ["transaction_create_date_time_eastern"], ["date_time_eastern"])
FNAME       = find_col(raw, ["patientfname","patient_first_name"], ["fname"])
LNAME       = find_col(raw, ["patientlname","patient_last_name"], ["lname"])
DOB         = find_col(raw, ["dateofbirth","date_of_birth","dob"], ["birth"])
PHONE       = find_col(raw, ["phone","patient_phone"])
INCIDENT    = find_col(raw, ["911_id"], ["911_id"])
REC_ID      = find_col(raw, ["id"])
PID         = find_col(raw, ["personal_id_number"], ["personal_id"])
MARKET      = find_col(raw, ["market_name","market"], ["market"])
DISPO       = find_col(raw, ["transaction_response_names"], ["response_name"])
CALLER_TYPE = find_col(raw, ["caller_type"], ["caller_type"])
NOTES       = find_col(raw, ["nurses_notes","nurse_notes","notes"], ["note"])

df = raw.copy()
df["call_dt"] = pd.to_datetime(df[DATE], errors="coerce")
no_date = int(df["call_dt"].isna().sum())
df = df[df["call_dt"].notna()].sort_values("call_dt").reset_index(drop=True)
df["record_id"] = df[REC_ID].astype(str) if REC_ID else df.index.astype(str)
print(f"{len(raw):,} calls loaded; {no_date:,} without a valid call date removed; {len(df):,} remain")
print(f"call dates: {df['call_dt'].min():%Y-%m-%d} to {df['call_dt'].max():%Y-%m-%d}")
fields = pd.DataFrame({"field":["call date","patient first name","patient last name","date of birth","patient phone",
                                "911 incident","record","personal identification number","market","disposition",
                                "caller type","nurse notes"],
                       "resolved":[DATE,FNAME,LNAME,DOB,PHONE,INCIDENT,REC_ID,PID,MARKET,DISPO,CALLER_TYPE,NOTES]})
fields.columns = ["Field","Source column"]
keep(fields, "field_mapping")
fields

## 3. Identity field profile

Fill rate and distinct values for each field used to identify a person. A high count on the most common value indicates a placeholder entry (for example, a default date of birth or a facility phone).

In [ ]:
BLANKS = {"", "nan", "none", "null", "na", "n/a", "unknown", "unk", "nat"}
def is_blank(s): return s.isna() | s.astype(str).str.strip().str.lower().isin(BLANKS)

prof = []
for label, col in [("patient first name",FNAME),("patient last name",LNAME),("date of birth",DOB),
                   ("patient phone",PHONE),("911 incident",INCIDENT),("personal identification number",PID)]:
    if col:
        b = is_blank(df[col])
        vc = df.loc[~b, col].astype(str).str.strip().str.lower().value_counts()
        prof.append({"Field":label, "Source column":col, "Share filled":round((~b).mean()*100,1),
                     "Distinct values":int(vc.size), "Calls on the most common value":int(vc.iloc[0]) if len(vc) else 0})
profile = pd.DataFrame(prof)
keep(profile, "identity_profile")
profile

## 4. Remove duplicate rows for the same 911 incident

Rows that share a 911 incident number describe the same incident. The earliest row is kept so a single incident is not counted as a repeat of itself.

In [ ]:
before = len(df)
if INCIDENT:
    has_inc = ~is_blank(df[INCIDENT])
    dup = has_inc & df[INCIDENT].astype(str).str.strip().duplicated(keep="first")
    df = df[~dup].reset_index(drop=True)
removed_dup = before - len(df)
print(f"duplicate incident rows removed: {removed_dup:,}; calls remaining: {len(df):,}")
if INCIDENT and removed_dup == 0:
    print("Every 911 incident number is unique, so this field identifies a row rather than grouping repeat transactions on one incident. Confirm with the source system owner whether multiple transactions on one 911 call can occur.")

## 5. Standardize names, date of birth, and phone

Names are lowercased with punctuation and suffixes (jr, sr, ii, iii, iv) removed. Placeholder names, placeholder dates of birth, and phones shared by many different people (for example, a facility line) are excluded from linking.

In [ ]:
SUFFIXES = {"jr","sr","ii","iii","iv"}
BAD_FIRST = {"unknown","unk","patient","pt","na","none","test","nan","refused","anonymous","caller","baby","infant","male","female"}
BAD_LAST  = {"doe","unknown","unk","patient","na","none","test","nan","refused","anonymous","caller"}
NICKNAMES = {"bob":"robert","bobby":"robert","rob":"robert","robby":"robert","bill":"william","billy":"william","will":"william",
    "liz":"elizabeth","beth":"elizabeth","betty":"elizabeth","jim":"james","jimmy":"james","mike":"michael","dave":"david",
    "tom":"thomas","tommy":"thomas","tony":"anthony","joe":"joseph","joey":"joseph","kate":"katherine","kathy":"katherine",
    "cathy":"catherine","chris":"christopher","dan":"daniel","danny":"daniel","steve":"steven","rick":"richard","ricky":"richard",
    "dick":"richard","sue":"susan","peggy":"margaret","maggie":"margaret","jen":"jennifer","jenny":"jennifer","larry":"lawrence",
    "patty":"patricia","ed":"edward","eddie":"edward","ted":"edward","charlie":"charles","chuck":"charles","sam":"samuel",
    "alex":"alexander","nick":"nicholas","matt":"matthew","andy":"andrew","greg":"gregory","ron":"ronald","don":"donald",
    "ken":"kenneth","jerry":"gerald","debbie":"deborah","deb":"deborah","vicky":"victoria","becky":"rebecca","terry":"terrence"}

def name_tokens(x):
    if pd.isna(x): return []
    return [t for t in re.sub(r"[^a-z ]", " ", str(x).lower()).split() if t not in SUFFIXES]

df["fn"] = df[FNAME].apply(lambda x: (name_tokens(x) or [""])[0]) if FNAME else ""
df["ln"] = df[LNAME].apply(lambda x: "".join(name_tokens(x))) if LNAME else ""
df["fn_canon"] = df["fn"].map(lambda f: NICKNAMES.get(f, f))
df["name_ok"] = (df["fn"].str.len() >= 2) & (df["ln"].str.len() >= 2) & ~df["fn"].isin(BAD_FIRST) & ~df["ln"].isin(BAD_LAST)

dob = pd.to_datetime(df[DOB], errors="coerce") if DOB else pd.Series(pd.NaT, index=df.index)
df["dob"] = dob.dt.strftime("%Y-%m-%d").fillna("")
df["dob_ok"] = dob.notna() & (dob.dt.year >= 1900) & (dob <= df["call_dt"])
ln_per_dob = df[df["dob_ok"] & df["name_ok"]].groupby("dob")["ln"].nunique()
placeholder_dobs = set(ln_per_dob[ln_per_dob > MAX_LAST_NAMES_PER_DOB].index) | {"1900-01-01","1901-01-01"}
df.loc[df["dob"].isin(placeholder_dobs), "dob_ok"] = False

def norm_phone(x):
    if pd.isna(x): return ""
    s = str(int(x)) if isinstance(x, (int, float, np.integer, np.floating)) else str(x)
    d = re.sub(r"\D", "", s)
    if len(d) == 11 and d[0] == "1": d = d[1:]
    ok = len(d) == 10 and d[0] not in "01" and len(set(d)) > 2 and d != "1234567890"
    return d if ok else ""

df["phone_n"] = df[PHONE].apply(norm_phone) if PHONE else ""
df["phone_ok"] = df["phone_n"] != ""
person_guess = df["fn_canon"] + "|" + df["ln"] + "|" + df["dob"]
ppl_per_phone = df[df["phone_ok"] & df["name_ok"]].assign(pg=person_guess).groupby("phone_n")["pg"].nunique()
shared_phones = set(ppl_per_phone[ppl_per_phone > MAX_PEOPLE_PER_PHONE].index)
df.loc[df["phone_n"].isin(shared_phones), "phone_ok"] = False

df["identifiable"] = df["name_ok"] & (df["dob_ok"] | df["phone_ok"])
print(f"placeholder dates of birth excluded: {len(placeholder_dobs):,}")
print(f"shared phones excluded (more than {MAX_PEOPLE_PER_PHONE} different people): {len(shared_phones):,}")

top_dob = (df[df["dob"] != ""].groupby("dob").agg(calls=("dob","size"), distinct_last_names=("ln","nunique"))
           .sort_values("distinct_last_names", ascending=False).head(10))
top_dob["excluded_as_placeholder"] = top_dob.index.isin(placeholder_dobs)
display(top_dob)

quality = pd.DataFrame({
    "Measure":["Calls after removing duplicate incidents","Calls with a usable first and last name","Calls with a usable date of birth",
               "Calls with a usable phone that is not shared","Calls we can tie to a person","Calls we cannot tie to a person"],
    "Calls":[len(df), int(df["name_ok"].sum()), int(df["dob_ok"].sum()), int(df["phone_ok"].sum()),
             int(df["identifiable"].sum()), int((~df["identifiable"]).sum())]})
quality["Share of calls"] = (quality["Calls"] / len(df) * 100).round(1)
keep(quality, "identity_quality")
quality

## 6. Link calls to persons

Rules are applied in order. Each rule links calls that the earlier rules did not already join.

| Rule | Link condition |
|---|---|
| 1 | Same first name, last name, and date of birth |
| 2 | Same last name and date of birth; first name matches by nickname, by spelling similarity with the same first letter, or one is the start of the other (Chris, Christopher) |
| 3 | Same first name and date of birth; last name matches by spelling similarity or one contains the other (hyphenated names) |
| 4 | Same date of birth and phone; first name matches under the rule 2 test, last name may differ (covers a last-name change) |
| 5 | Same first name, last name, and phone, where at most one valid date of birth exists among those calls |

A person is capped at 3 distinct first names and 3 distinct last names. A fuzzy link that would push a person past either cap is rejected and counted in the Match Rules table. Without that cap, a chain of individually reasonable links can merge many different people who share a date of birth: A links to B, B links to C, and A and C end up as one person even though their names are not alike. The cap trades a small number of missed links for protection against that. Rule 1 never adds name variety, because it requires identical names.

Spelling similarity is the Jaro-Winkler score, a standard name-comparison measure from 0 to 1; the threshold is 0.88 (for example, smith and smyth score 0.89). Names are never linked on name alone. Calls with the same name but different valid dates of birth stay separate (for example, a parent and child with the same name).

In [ ]:
ident = df[df["identifiable"]].reset_index(drop=True)
n = len(ident)
parent = np.arange(n)
fn_sets = [{v} for v in ident["fn"]]
ln_sets = [{v} for v in ident["ln"]]
def find(x):
    root = x
    while parent[root] != root: root = parent[root]
    while parent[x] != root: parent[x], x = root, parent[x]
    return root

T0 = "0 personal identification number"
T1 = "1 exact name and date of birth"
T2 = "2 similar first name, same last name and date of birth"
T3 = "3 similar last name, same first name and date of birth"
T4 = "4 same first name, date of birth, and phone"
T5 = "5 same name and phone, one date of birth or none"
merges = {t: 0 for t in [T0,T1,T2,T3,T4,T5]}
blocked = {t: 0 for t in [T0,T1,T2,T3,T4,T5]}
qa_links = []
def union(a, b, tier, guard=False):
    ra, rb = find(a), find(b)
    if ra == rb: return
    fns, lns = fn_sets[ra] | fn_sets[rb], ln_sets[ra] | ln_sets[rb]
    if guard and (len(fns) > MAX_FIRST_NAME_VARIANTS or len(lns) > MAX_LAST_NAME_VARIANTS):
        blocked[tier] += 1; return
    parent[rb] = ra; fn_sets[ra], ln_sets[ra] = fns, lns; merges[tier] += 1
    if tier != T1: qa_links.append((a, b, tier))

def sim(a, b):
    if a == b: return 1.0
    la, lb = len(a), len(b)
    if not la or not lb: return 0.0
    r = max(max(la, lb) // 2 - 1, 0)
    ma, mb, m = [False]*la, [False]*lb, 0
    for i, ch in enumerate(a):
        for j in range(max(0, i - r), min(lb, i + r + 1)):
            if not mb[j] and b[j] == ch:
                ma[i] = mb[j] = True; m += 1; break
    if not m: return 0.0
    t, k = 0, 0
    for i in range(la):
        if ma[i]:
            while not mb[k]: k += 1
            if a[i] != b[k]: t += 1
            k += 1
    jaro = (m / la + m / lb + (m - t / 2) / m) / 3
    p = 0
    for x, y in zip(a[:4], b[:4]):
        if x != y: break
        p += 1
    return jaro + p * 0.1 * (1 - jaro)
def first_match(a, b):
    if NICKNAMES.get(a, a) == NICKNAMES.get(b, b): return True
    if a[:1] != b[:1]: return False
    short, long_ = sorted([a, b], key=len)
    return sim(a, b) >= NAME_MIN_SIMILARITY or (len(short) >= 3 and long_.startswith(short))
def last_match(a, b):
    short, long_ = sorted([a, b], key=len)
    return sim(a, b) >= NAME_MIN_SIMILARITY or (len(short) >= 4 and short in long_)

if USE_PERSONAL_ID and PID:
    pid_ok = ~is_blank(ident[PID])
    for g in ident[pid_ok].groupby(ident.loc[pid_ok, PID].astype(str).str.strip()).groups.values():
        g = list(g)
        for j in g[1:]: union(g[0], j, T0)

has_dob = ident["dob_ok"]
for g in ident[has_dob].groupby(["fn","ln","dob"]).groups.values():
    g = list(g)
    for j in g[1:]: union(g[0], j, T1)

def fuzzy_pass(frame, block_cols, compare_col, matcher, tier):
    multi = frame.groupby(block_cols)[compare_col].transform("nunique") > 1
    for _, g in frame[multi].groupby(block_cols):
        reps = g.drop_duplicates(compare_col)
        idx, vals = list(reps.index), list(reps[compare_col])
        for i in range(len(idx)):
            for j in range(i + 1, len(idx)):
                if matcher(vals[i], vals[j]): union(idx[i], idx[j], tier, guard=True)

fuzzy_pass(ident[has_dob], ["ln","dob"], "fn", first_match, T2)
fuzzy_pass(ident[has_dob], ["fn_canon","dob"], "ln", last_match, T3)
ident["full_name"] = ident["fn"] + "|" + ident["ln"]
fuzzy_pass(ident[has_dob & ident["phone_ok"]], ["dob","phone_n"], "full_name",
           lambda a, b: first_match(a.split("|")[0], b.split("|")[0]), T4)

np_frame = ident[ident["phone_ok"]]
np_frame = np_frame[np_frame.groupby(["fn_canon","ln","phone_n"])["fn"].transform("size") > 1]
for _, g in np_frame.groupby(["fn_canon","ln","phone_n"]):
    if g.loc[g["dob_ok"], "dob"].nunique() <= 1:
        idx = list(g.index)
        for j in idx[1:]: union(idx[0], j, T5, guard=True)

ident["cluster"] = [find(i) for i in range(n)]
first_seen = ident.groupby("cluster")["call_dt"].min().sort_values()
key_map = {c: f"P{i+1:06d}" for i, c in enumerate(first_seen.index)}
ident["person_key"] = ident["cluster"].map(key_map)

tiers = pd.DataFrame({"Rule": list(merges.keys()), "Calls linked": list(merges.values()),
                      "Links rejected by the name limit": list(blocked.values())})
tiers = tiers[(tiers["Rule"] != T0) | USE_PERSONAL_ID]
keep(tiers, "match_rules")
n_persons = ident["person_key"].nunique()
print(f"identifiable calls: {n:,}; distinct persons: {n_persons:,}; average calls per person: {n/n_persons:.2f}")
t = tiers[tiers["Calls linked"] + tiers["Links rejected by the name limit"] > 0]
fig, ax = plt.subplots(figsize=(10, max(3, 0.6*len(t)+1)))
y = np.arange(len(t)); h = 0.38
ax.barh(y - h/2, t["Calls linked"], h, color=TEAL, label="calls linked")
ax.barh(y + h/2, t["Links rejected by the name limit"], h, color=CORAL, label="links rejected by the name limit")
ax.set_yticks(y); ax.set_yticklabels(t["Rule"], fontsize=9); ax.legend()
ax.set_title("Links created by each matching rule")
plt.tight_layout(); plt.show()
tiers

## 7. Link quality checks

A sample of links from the fuzzy rules (2 to 5), showing which fields agree and how similar the names are. Record numbers allow each pair to be looked up in the source data. Names are not displayed. The cluster check lists the largest persons by call count; a person with more than one distinct valid date of birth would indicate an incorrect link.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
qa = pd.DataFrame(qa_links, columns=["a","b","rule"])
if len(qa):
    per_rule = max(1, QA_SAMPLE_N // max(1, qa["rule"].nunique()))
    qa = pd.concat([g.sample(min(len(g), per_rule), random_state=RANDOM_SEED) for _, g in qa.groupby("rule")])
    A, B = ident.loc[qa["a"]].reset_index(drop=True), ident.loc[qa["b"]].reset_index(drop=True)
    qa_out = pd.DataFrame({
        "rule": qa["rule"].values,
        "record_a": A["record_id"], "record_b": B["record_id"],
        "first_name_similarity": [round(sim(x, y), 2) for x, y in zip(A["fn"], B["fn"])],
        "last_name_similarity": [round(sim(x, y), 2) for x, y in zip(A["ln"], B["ln"])],
        "same_date_of_birth": (A["dob"] == B["dob"]) & A["dob_ok"] & B["dob_ok"],
        "same_phone": (A["phone_n"] == B["phone_n"]) & A["phone_ok"] & B["phone_ok"],
        "days_apart": ((B["call_dt"] - A["call_dt"]).abs().dt.total_seconds() / 86400).round(1)})
else:
    qa_out = pd.DataFrame(columns=["rule","record_a","record_b"])
keep(qa_out, "match_qa_sample")
display(qa_out.head(30))

ident["dob_valid"] = ident["dob"].where(ident["dob_ok"])
ident["phone_valid"] = ident["phone_n"].where(ident["phone_ok"])
clus = ident.groupby("person_key").agg(**{"Calls":("record_id","size"), "First names":("fn","nunique"),
        "Last names":("ln","nunique"), "Dates of birth":("dob_valid","nunique"), "Phones":("phone_valid","nunique")})
multi_dob = int((clus["Dates of birth"] > 1).sum())
print(f"persons with more than one distinct valid date of birth: {multi_dob:,}")
largest = clus.sort_values("Calls", ascending=False).head(15).reset_index()
keep(largest, "largest_persons")
display(largest)

flags = pd.DataFrame({
    "more than one date of birth": clus["Dates of birth"] > 1,
    "at the first name limit": clus["First names"] >= MAX_FIRST_NAME_VARIANTS,
    "at the last name limit": clus["Last names"] >= MAX_LAST_NAME_VARIANTS,
    "many different phones": clus["Phones"] >= REVIEW_PHONES_PER_PERSON})
hit = flags.any(axis=1)
review = clus[hit].copy()
review["Review reason"] = ["; ".join(flags.columns[row]) for row in flags[hit].values]
review = review.sort_values("Calls", ascending=False).reset_index()
keep(review, "persons_needing_review")
print(f"persons flagged for review: {len(review):,} ({len(review)/len(clus)*100:.2f}% of persons), "
      f"covering {int(review['Calls'].sum()):,} calls")
print("These are not errors. They are the persons whose links rest on the weakest evidence, and they should be checked in the source data before any list of frequent callers is shared.")
display(review.head(15))

## 8. Behavioral health keyword flag

Each call is flagged with the same whole-word keyword list and negation handling used in the behavioral health screen, so repeat rates can be compared for calls with and without behavioral health language. The flag is keyword-based and is not validated in this notebook.

In [ ]:
kw_path = os.path.join(PROMPT_DIR, "bh_keywords.txt")
NEGATIONS = ["denies","denied","no ","without","negative for","ruled out","not ","non-"]
if NOTES and os.path.exists(kw_path):
    with open(kw_path) as f:
        kws = [k.strip().lower() for k in f.read().splitlines() if k.strip() and not k.strip().startswith("#")]
    pats = [re.compile("(?<![a-z0-9])" + re.escape(k) + "(?:s|es)?(?![a-z0-9])") for k in kws]
    def kw_hit(text):
        t = str(text).lower()
        for p in pats:
            for m in p.finditer(t):
                if not any(neg in t[max(0, m.start()-15):m.start()] for neg in NEGATIONS): return True
        return False
    ident["bh_flag"] = ident[NOTES].fillna("").apply(kw_hit)
    print(f"keywords loaded: {len(kws)}; identifiable calls with behavioral health keyword matches: {int(ident['bh_flag'].sum()):,} ({ident['bh_flag'].mean()*100:.1f}%)")
else:
    ident["bh_flag"] = False
    print("keyword file or notes column not found; behavioral health flag not applied")

## 9. Repeat calls, timing, and rates

A repeat call is any call that is a person's second or later call. This section reports how many repeat calls there were, how soon each came after the person's previous call, and the share of calls that were followed by a repeat within 30, 60, and 90 days.

In [ ]:
d = ident.sort_values(["person_key","call_dt"]).reset_index(drop=True)
g = d.groupby("person_key")["call_dt"]
d["next_gap_days"] = (g.shift(-1) - d["call_dt"]).dt.total_seconds() / 86400
d["prev_gap_days"] = (d["call_dt"] - g.shift(1)).dt.total_seconds() / 86400
d["is_repeat"] = d["prev_gap_days"].notna()
d["same_calendar_day"] = d["is_repeat"] & (d["call_dt"].dt.normalize() == g.shift(1).dt.normalize())
if DISPO: d["prev_disposition"] = d.groupby("person_key")[DISPO].shift(1)
data_end = d["call_dt"].max()
for w in WINDOWS:
    d[f"eligible_{w}"] = d["call_dt"] <= data_end - pd.Timedelta(days=w)
    d[f"return_{w}"] = d["next_gap_days"].le(w)

rows = []
for w in WINDOWS:
    e = d[d[f"eligible_{w}"]]
    persons_e = e["person_key"].nunique()
    persons_r = e.loc[e[f"return_{w}"], "person_key"].nunique()
    rows.append({"Window": f"{w} days", "Calls measured": len(e), "Calls with a return": int(e[f"return_{w}"].sum()),
                 "Return rate": round(e[f"return_{w}"].mean()*100, 1),
                 "People measured": persons_e, "People with a return": persons_r,
                 "Share of people with a return": round(persons_r / persons_e * 100, 1) if persons_e else 0})
rates = pd.DataFrame(rows)
keep(rates, "return_rates")

fig, ax = plt.subplots(figsize=(8,4))
ax.bar(rates["Window"], rates["Return rate"], color=[TEAL, NAVY, GREY])
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title("Share of calls followed by a repeat call from the same person")
for i, v in enumerate(rates["Return rate"]): ax.annotate(f"{v:.1f}%", (i, v), xytext=(0,4), textcoords="offset points", ha="center")
plt.tight_layout(); plt.show()
rates

In [ ]:
n_repeat = int(d["is_repeat"].sum())
n_same_day = int(d["same_calendar_day"].sum())
print(f"Repeat calls: {n_repeat:,} of {len(d):,} calls tied to a person ({n_repeat/len(d)*100:.1f}%)")
print(f"Repeat calls on the same calendar day as the previous call: {n_same_day:,} ({n_same_day/n_repeat*100:.1f}% of repeat calls)")
print(f"Repeat calls on a later calendar day: {n_repeat-n_same_day:,} ({(n_repeat-n_same_day)/n_repeat*100:.1f}% of repeat calls)")
totals = pd.DataFrame({
    "Measure":["Calls tied to a person","People","First calls","Repeat calls",
               "Repeat calls as a share of calls tied to a person","Repeat calls as a share of all calls loaded"],
    "Figure":[len(d), d["person_key"].nunique(), len(d)-n_repeat, n_repeat,
              round(n_repeat/len(d)*100,1), round(n_repeat/len(raw)*100,1)]})
keep(totals, "repeat_call_totals")
display(totals)
same_day = pd.DataFrame({"Timing":["Same calendar day as the previous call","A later calendar day"],
                         "Repeat calls":[n_same_day, n_repeat-n_same_day]})
same_day["Share of repeat calls"] = (same_day["Repeat calls"]/n_repeat*100).round(1)
keep(same_day, "same_day_split")
same_day

In [ ]:
BINS   = [-0.001, 1, 3, 5, 7, 14, 30, 60, 90, 180, np.inf]
LABELS = ["1 day or less","2-3 days","4-5 days","6-7 days","8-14 days","15-30 days",
          "31-60 days","61-90 days","91-180 days","over 180 days"]
gap = pd.cut(d.loc[d["is_repeat"], "prev_gap_days"], bins=BINS, labels=LABELS).value_counts().reindex(LABELS)
gap_df = gap.rename("Repeat calls").to_frame()
gap_df["Share of repeat calls"] = (gap_df["Repeat calls"]/n_repeat*100).round(1)
gap_df.index.name = "Days since the previous call"
keep(gap_df.reset_index(), "repeat_call_timing")

fig, ax = plt.subplots(figsize=(11,4))
ax.bar(range(len(gap_df)), gap_df["Repeat calls"], color=[CORAL,CORAL,CORAL,CORAL,TEAL,TEAL,GREY,GREY,GREY,GREY])
ax.set_xticks(range(len(gap_df))); ax.set_xticklabels(LABELS, fontsize=9, rotation=20)
ax.set_title("Days from a repeat call back to that person's previous call")
for i, v in enumerate(gap_df["Repeat calls"]): ax.annotate(f"{v:,}", (i, v), xytext=(0,4), textcoords="offset points", ha="center", fontsize=9)
plt.tight_layout(); plt.show()
gap_df

The short-window view the review asked for: how many repeat calls came back within three, five, and seven days of the previous call, alongside the longer windows.

In [ ]:
CUM_DAYS = [1, 3, 5, 7, 14, 30, 60, 90]
rows = []
for k in CUM_DAYS:
    n = int((d.loc[d["is_repeat"], "prev_gap_days"] <= k).sum())
    n_later = int(((d["prev_gap_days"] <= k) & (~d["same_calendar_day"]) & d["is_repeat"]).sum())
    rows.append({"Within": f"{k} day" if k == 1 else f"{k} days",
                 "Repeat calls": n, "Share of repeat calls": round(n/n_repeat*100, 1),
                 "Repeat calls excluding same calendar day": n_later,
                 "Share of repeat calls excluding same calendar day": round(n_later/(n_repeat-n_same_day)*100, 1)})
cum = pd.DataFrame(rows)
keep(cum, "repeat_call_cumulative")

fig, ax = plt.subplots(figsize=(10,4))
ax.bar(cum["Within"], cum["Repeat calls"], color=TEAL)
ax.set_title("Repeat calls that came back within each window")
for i, v in enumerate(cum["Repeat calls"]): ax.annotate(f"{v:,}", (i, v), xytext=(0,4), textcoords="offset points", ha="center", fontsize=9)
plt.tight_layout(); plt.show()
display(cum)
print("Windows are cumulative. The last two columns repeat the count with same-calendar-day repeats removed.")

## 10. Calls per person and frequent callers

Frequent callers are counted by the highest number of calls a person made inside any rolling 30, 60, or 90 day period.

In [ ]:
per = d.groupby("person_key").agg(calls=("call_dt","size"), first_call=("call_dt","min"), last_call=("call_dt","max"),
                                  behavioral_health_keyword_calls=("bh_flag","sum"))
def max_in_window(t, w):
    s = np.sort(t.values.astype("datetime64[s]").astype(np.int64))
    return int((np.searchsorted(s, s + w*86400, side="right") - np.arange(len(s))).max())
repeaters = d[d["person_key"].isin(per.index[per["calls"] > 1])]
for w in WINDOWS:
    m = repeaters.groupby("person_key")["call_dt"].apply(lambda t: max_in_window(t, w))
    per[f"max_calls_in_{w}_days"] = m.reindex(per.index).fillna(1).astype(int)

buckets = pd.cut(per["calls"], bins=[0,1,2,4,9,np.inf], labels=["1","2","3-4","5-9","10 or more"])
cpp = per.groupby(buckets).agg(**{"People":("calls","size"), "Calls":("calls","sum")})
cpp["Share of people"] = (cpp["People"] / cpp["People"].sum() * 100).round(1)
cpp["Share of calls"] = (cpp["Calls"] / cpp["Calls"].sum() * 100).round(1)
cpp.index.name = "Calls per person"
keep(cpp.reset_index(), "calls_per_person")

ranked = per["calls"].sort_values(ascending=False).values
share_persons = np.arange(1, len(ranked)+1) / len(ranked) * 100
share_calls = np.cumsum(ranked) / ranked.sum() * 100
fig, ax = plt.subplots(figsize=(8,4.5))
ax.plot(share_persons, share_calls, lw=2, color=TEAL)
ax.plot([0,100],[0,100], lw=1, ls="--", color=GREY)
for mark in [1, 5, 10, 25]:
    v = share_calls[min(int(len(ranked)*mark/100), len(ranked)-1)]
    ax.annotate(f"top {mark}% of persons = {v:.0f}% of calls", (mark, v), xytext=(8,-10),
                textcoords="offset points", fontsize=8, color=NAVY)
    ax.plot([mark],[v], "o", color=NAVY, ms=4)
ax.set_xlabel("share of persons, most frequent first"); ax.set_ylabel("share of calls")
ax.xaxis.set_major_formatter(mtick.PercentFormatter()); ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title("How concentrated call volume is among callers")
plt.tight_layout(); plt.show()
print("The dashed line is what the curve would look like if every person called the same number of times.")

freq_rows = []
for w in WINDOWS:
    for k in FREQUENT_THRESHOLDS:
        sel = per[per[f"max_calls_in_{w}_days"] >= k]
        freq_rows.append({"Window and threshold": f"{w} days, {k} or more calls", "_w": w, "_k": k,
                          "People": len(sel), "Share of people": round(len(sel) / len(per) * 100, 2),
                          "Calls from these people": int(sel["calls"].sum()),
                          "Share of calls": round(sel["calls"].sum() / per["calls"].sum() * 100, 1)})
freq = pd.DataFrame(freq_rows)
keep(freq, "frequent_callers")

fig, ax = plt.subplots(figsize=(10,4))
piv = freq.pivot(index="_k", columns="_w", values="People")
x = np.arange(len(piv)); wd = 0.26
for i, w in enumerate(piv.columns):
    ax.bar(x + (i-1)*wd, piv[w], wd, color=[TEAL, NAVY, GREY][i % 3], label=f"{w} days")
ax.set_xticks(x); ax.set_xticklabels([f"{k} or more calls" for k in piv.index])
ax.set_ylabel("persons"); ax.legend(title="rolling window")
ax.set_title("Frequent callers by rolling window")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(9,4))
x = np.arange(len(cpp)); wd = 0.38
ax.bar(x - wd/2, cpp["Share of people"], wd, color=TEAL, label="share of people")
ax.bar(x + wd/2, cpp["Share of calls"], wd, color=NAVY, label="share of calls")
ax.set_xticks(x); ax.set_xticklabels(cpp.index.astype(str)); ax.set_xlabel("calls per person")
ax.yaxis.set_major_formatter(mtick.PercentFormatter()); ax.legend()
ax.set_title("Calls per person - share of people and share of calls")
plt.tight_layout(); plt.show()
display(cpp); display(freq)

In [ ]:
top = per.sort_values("calls", ascending=False).head(25).copy()
top = top.rename(columns={"calls":"Calls","first_call":"First call","last_call":"Last call",
                          "behavioral_health_keyword_calls":"Calls with behavioral health keywords",
                          "max_calls_in_30_days":"Most calls in any 30 days",
                          "max_calls_in_60_days":"Most calls in any 60 days",
                          "max_calls_in_90_days":"Most calls in any 90 days"})
top_rows = d[d["person_key"].isin(top.index)]
mode_of = lambda s: s.astype(str).mode().iloc[0] if len(s.dropna()) else ""
top_market = top_rows.groupby("person_key")[MARKET].agg(mode_of) if MARKET else None
top_dispo = top_rows.groupby("person_key")[DISPO].agg(mode_of) if DISPO else None
top["Days between first and last call"] = (top["Last call"] - top["First call"]).dt.days
if MARKET: top["Most common market"] = top_market.reindex(top.index)
if DISPO: top["Most common disposition"] = top_dispo.reindex(top.index)
top["First call"] = top["First call"].dt.strftime("%Y-%m-%d"); top["Last call"] = top["Last call"].dt.strftime("%Y-%m-%d")
top = top.reset_index().rename(columns={"person_key":"Person"})
rmap = review.set_index("person_key")["Review reason"] if len(review) else pd.Series(dtype=object)
top["Needs review"] = top["Person"].isin(rmap.index)
top["Review reason"] = top["Person"].map(rmap).fillna("")
keep(top, "top_frequent_callers")
top

## 11. Share of calls followed by a repeat, by group

These figures describe the call that was followed by a repeat, not the repeat call itself. The disposition shown is the one recorded on that earlier call. Section 12 reports the disposition recorded on the repeat call.

Each rate uses its own eligible index calls, so market and disposition rates are normalized to their own call volume.

In [ ]:
def return_by(col, label, top_n=None):
    key = d[col].fillna("(blank)").astype(str) if col in d.columns else pd.Series("(blank)", index=d.index)
    out = None
    for w in WINDOWS:
        e = d[f"eligible_{w}"]
        t = d[e].groupby(key[e]).agg(**{f"Calls measured ({w} days)": (f"return_{w}","size"),
                                        f"Return rate ({w} days)": (f"return_{w}","mean")})
        t[f"Return rate ({w} days)"] = (t[f"Return rate ({w} days)"] * 100).round(1)
        out = t if out is None else out.join(t, how="outer")
    out = out.sort_values(f"Calls measured ({WINDOWS[0]} days)", ascending=False)
    if top_n: out = out.head(top_n)
    out.index.name = label
    return out.reset_index()

def rate_chart(t, label, title):
    o = t.sort_values(f"Return rate ({WINDOWS[0]} days)")
    fig, ax = plt.subplots(figsize=(10, max(3, 0.35*len(o) + 1)))
    ax.barh(range(len(o)), o[f"Return rate ({WINDOWS[0]} days)"], color=TEAL)
    ax.set_yticks(range(len(o))); ax.set_yticklabels([str(v)[:45] for v in o[label]], fontsize=9)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter()); ax.set_title(title)
    for i, v in enumerate(o[f"Return rate ({WINDOWS[0]} days)"]):
        ax.annotate(f"{v:.1f}%", (v, i), xytext=(4,0), textcoords="offset points", va="center", fontsize=8)
    plt.tight_layout(); plt.show()

by_market = keep(return_by(MARKET, "Market"), "by_market") if MARKET else None
if by_market is not None: rate_chart(by_market, "Market", f"{WINDOWS[0]}-day return rate by market"); display(by_market)

In [ ]:
by_dispo = keep(return_by(DISPO, "Disposition", top_n=15), "by_disposition") if DISPO else None
if by_dispo is not None: rate_chart(by_dispo, "Disposition", f"{WINDOWS[0]}-day rate by disposition of the call that was followed by a repeat (15 most common)"); display(by_dispo)

In [ ]:
by_caller = keep(return_by(CALLER_TYPE, "Caller type"), "by_caller_type") if CALLER_TYPE else None
if by_caller is not None: display(by_caller)
d["behavioral_health_keyword_match"] = np.where(d["bh_flag"], "keyword match", "no keyword match")
by_bh = keep(return_by("behavioral_health_keyword_match", "Behavioral health keywords"), "by_behavioral_health")
display(by_bh)
fig, ax = plt.subplots(figsize=(9,4))
x = np.arange(len(by_bh)); wd = 0.26
for i, w in enumerate(WINDOWS):
    ax.bar(x + (i-1)*wd, by_bh[f"Return rate ({w} days)"], wd, color=[TEAL, NAVY, GREY][i % 3], label=f"{w} days")
ax.set_xticks(x); ax.set_xticklabels(by_bh["Behavioral health keywords"])
ax.yaxis.set_major_formatter(mtick.PercentFormatter()); ax.legend(title="window")
ax.set_title("Return rate for calls with and without behavioral-health keywords")
plt.tight_layout(); plt.show()
print("The keyword match is not validated in this notebook, so read this as a direction rather than a measured difference.")

## 12. Disposition of the repeat call, and what came before it

The earlier breakdowns report the disposition recorded on the call that was followed by a repeat. This section reports the disposition recorded on the repeat call itself, and the disposition of that person's previous call.

In [ ]:
if DISPO:
    rep = d[d["is_repeat"]].copy()
    rep["Disposition of the repeat call"] = rep[DISPO].fillna("(blank)").astype(str)
    rep["Disposition of the previous call"] = rep["prev_disposition"].fillna("(blank)").astype(str)
    rd = rep["Disposition of the repeat call"].value_counts().rename("Repeat calls").to_frame()
    rd["Share of repeat calls"] = (rd["Repeat calls"]/len(rep)*100).round(1)
    rd.index.name = "Disposition of the repeat call"
    keep(rd.reset_index(), "repeat_call_disposition")
    o = rd.head(10).sort_values("Repeat calls")
    fig, ax = plt.subplots(figsize=(10, max(3, 0.34*len(o)+1)))
    ax.barh(range(len(o)), o["Repeat calls"], color=TEAL)
    ax.set_yticks(range(len(o))); ax.set_yticklabels(o.index, fontsize=9)
    ax.set_title("Disposition recorded on the repeat call")
    for i,v in enumerate(o["Repeat calls"]): ax.annotate(f"{v:,}",(v,i),xytext=(4,0),textcoords="offset points",va="center",fontsize=9)
    plt.tight_layout(); plt.show()
    display(rd)

In [ ]:
if DISPO:
    top_d = rd.head(8).index.tolist()
    m = rep[rep["Disposition of the repeat call"].isin(top_d) & rep["Disposition of the previous call"].isin(top_d)]
    flow = pd.crosstab(m["Disposition of the previous call"], m["Disposition of the repeat call"])
    flow = flow.reindex(index=top_d, columns=top_d).fillna(0).astype(int)
    keep(flow.reset_index(), "disposition_flow_counts")
    row_pct = (flow.div(flow.sum(axis=1).replace(0, np.nan), axis=0)*100).round(1)
    keep(row_pct.reset_index(), "disposition_flow_row_percent")
    print("Rows are the previous call. Columns are the repeat call. Row percentages show where each starting disposition went.")
    display(flow); display(row_pct)

Where the repeat calls that ended in each disposition started. The virtual care view is broken out separately because it was raised in review.

In [ ]:
if DISPO:
    col_pct = (flow.div(flow.sum(axis=0).replace(0, np.nan), axis=1)*100).round(1)
    keep(col_pct.reset_index(), "disposition_flow_origin_percent")
    display(col_pct)
    for target in [c for c in ["VIRTUAL CARE","NN ER","BLS"] if c in col_pct.columns]:
        o = col_pct[target].dropna().sort_values()
        o = o[o > 0]
        fig, ax = plt.subplots(figsize=(9, max(2.6, 0.34*len(o)+1)))
        ax.barh(range(len(o)), o.values, color=TEAL)
        ax.set_yticks(range(len(o))); ax.set_yticklabels(o.index, fontsize=9)
        ax.xaxis.set_major_formatter(mtick.PercentFormatter())
        ax.set_title(f"Previous-call disposition for repeat calls dispositioned {target}")
        for i,v in enumerate(o.values): ax.annotate(f"{v:.1f}%",(v,i),xytext=(4,0),textcoords="offset points",va="center",fontsize=9)
        plt.tight_layout(); plt.show()
        n_t = int(flow[target].sum())
        print(f"{target}: {n_t:,} repeat calls with a previous call in the eight most common dispositions.")

In [ ]:
if DISPO and "VIRTUAL CARE" in rep["Disposition of the repeat call"].unique():
    vc = rep[rep["Disposition of the repeat call"] == "VIRTUAL CARE"]
    vo = vc["Disposition of the previous call"].value_counts().rename("Repeat calls").to_frame()
    vo["Share of these repeat calls"] = (vo["Repeat calls"]/len(vc)*100).round(1)
    vo.index.name = "Disposition of the previous call"
    keep(vo.reset_index(), "virtual_care_origin")
    o = vo.head(10).sort_values("Repeat calls")
    fig, ax = plt.subplots(figsize=(10, max(3, 0.34*len(o)+1)))
    ax.barh(range(len(o)), o["Repeat calls"], color=TEAL)
    ax.set_yticks(range(len(o))); ax.set_yticklabels(o.index, fontsize=9)
    ax.set_title("Previous-call disposition for every repeat call dispositioned VIRTUAL CARE")
    for i,v in enumerate(o["Repeat calls"]): ax.annotate(f"{v:,}",(v,i),xytext=(4,0),textcoords="offset points",va="center",fontsize=9)
    plt.tight_layout(); plt.show()
    display(vo)
    print(f"{len(vc):,} repeat calls were dispositioned VIRTUAL CARE. This table covers all of them, not only the eight most common previous dispositions.")

## 13. Monthly trend

The 30-day return rate by month of the index call. Months without a full 30-day follow-up window before the end of the data are excluded.

In [ ]:
w = WINDOWS[0]
d["month"] = d["call_dt"].dt.to_period("M")
full = d["month"].dt.end_time <= data_end - pd.Timedelta(days=w)
trend = d[full].groupby("month").agg(**{"Calls measured":(f"return_{w}","size"), "Calls with a return":(f"return_{w}","sum")})
trend["Return rate"] = (trend["Calls with a return"] / trend["Calls measured"] * 100).round(1)
trend.index = trend.index.astype(str); trend.index.name = "Month"
keep(trend.reset_index(), "return_trend")
fig, ax = plt.subplots(figsize=(12,4))
ax.plot(range(len(trend)), trend["Return rate"], marker="o", lw=2, color=TEAL)
ax.set_xticks(range(len(trend))); ax.set_xticklabels(trend.index, rotation=60, fontsize=8)
ax.set_ylim(0, max(10, trend["Return rate"].max() * 1.2) if len(trend) else 10)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title(f"{w}-day return rate by month of the first call"); ax.set_ylabel("% of calls")
plt.tight_layout(); plt.show()
trend

## 14. Write the Excel output

The workbook carries ten tabs. Data Info explains how every number was derived and defines each column on every other tab. Repeat Calls and Repeat Timing answer how many repeat calls there were and how soon they came back. Disposition Flow and Virtual Care Origin answer where the repeat calls ended and where they started. Caller Frequency, Monthly Trend, By Group, Frequent Callers, and Method and Quality carry the remaining detail.

In [ ]:
w30 = WINDOWS[0]
multi = per[per["calls"] > 1]
summary_lines = [
    f"Run {RUN_ID}. Source file: {SOURCE_FILE}.",
    f"Calls loaded: {len(raw):,}. Calls after removing rows without a date and duplicate incident rows: {len(df):,}.",
    f"Call dates: {df['call_dt'].min():%Y-%m-%d} to {df['call_dt'].max():%Y-%m-%d}.",
    f"Identifiable calls: {len(ident):,} ({len(ident)/len(df)*100:.1f}% of calls). Distinct persons: {n_persons:,}.",
    f"Persons with more than one call: {len(multi):,} ({len(multi)/len(per)*100:.1f}% of persons), accounting for "
    f"{int(multi['calls'].sum()):,} calls ({multi['calls'].sum()/per['calls'].sum()*100:.1f}% of identifiable calls).",
]
for _, r in rates.iterrows():
    summary_lines.append(
        f"Within {r['Window']}: {r['Return rate']}% of calls were followed by another call "
        f"from the same person, and {r['Share of people with a return']}% of people had at least one return.")
summary_lines.append(
    f"Repeat calls: {n_repeat:,} of {len(d):,} calls tied to a person ({n_repeat/len(d)*100:.1f}%). "
    f"{n_same_day:,} were on the same calendar day as the previous call.")
for k in [3, 5, 7, 30]:
    n = int((d.loc[d['is_repeat'], 'prev_gap_days'] <= k).sum())
    summary_lines.append(f"Repeat calls within {k} days of the previous call: {n:,} ({n/n_repeat*100:.1f}% of repeat calls).")
summary_lines.append(f"Persons flagged for review before sharing any caller list: {len(review):,}.")
for line in summary_lines: print(line)

def stack(parts):
    out = []
    for label, frame in parts:
        if frame is None or not len(frame): continue
        f = frame.copy(); f.columns = [str(c) for c in f.columns]
        f = f.rename(columns={f.columns[0]: "Item"})
        long = f.melt(id_vars=["Item"], var_name="Measure", value_name="Value")
        long.insert(0, "Section", label)
        long["Item"] = long["Item"].astype(str)
        out.append(long[["Section","Item","Measure","Value"]])
    if not out: return pd.DataFrame()
    r = pd.concat(out, ignore_index=True)
    r = r[r["Value"].notna()]
    return r[r["Value"].astype(str).str.strip() != ""].reset_index(drop=True)

def sanitize(x):
    o = x.copy(); o.columns = [str(c) for c in o.columns]
    for c in o.columns:
        if o[c].dtype == object or str(o[c].dtype).startswith("category"):
            o[c] = o[c].apply(lambda v: "" if v is None or (isinstance(v, float) and pd.isna(v)) else str(v))
    return o

G = RESULTS.get
repeat_tab = stack([("Repeat calls in total", G("repeat_call_totals")),
                    ("Same calendar day or later", G("same_day_split")),
                    ("Share of calls followed by a repeat", G("return_rates"))])
timing_tab = stack([("Repeat calls by window, cumulative", G("repeat_call_cumulative")),
                    ("Days since the previous call", G("repeat_call_timing"))])
fq = G("frequent_callers")
if fq is not None and len(fq): fq = fq.drop(columns=["_w","_k"])
frequency_tab = stack([("Calls per person", G("calls_per_person")),
                       ("Frequent callers by rolling window", fq)])
trend_tab = G("return_trend")
group_tab = stack([("Market", G("by_market")), ("Disposition of the call before the repeat", G("by_disposition")),
                   ("Caller type", G("by_caller_type")), ("Behavioral health keywords", G("by_behavioral_health"))])
flow_tab = stack([("Disposition of the repeat call", G("repeat_call_disposition")),
                  ("Previous call to repeat call, counts", G("disposition_flow_counts")),
                  ("Previous call to repeat call, row percent", G("disposition_flow_row_percent")),
                  ("Where each repeat-call disposition came from, percent", G("disposition_flow_origin_percent"))])
origin_tab = G("virtual_care_origin")
method_tab = stack([("Field mapping", G("field_mapping")), ("Identity field profile", G("identity_profile")),
                    ("Identity quality", G("identity_quality")), ("Matching rules", G("match_rules"))])
OUTPUT = [("Data Info", None), ("Repeat Calls", repeat_tab), ("Repeat Timing", timing_tab),
          ("Disposition Flow", flow_tab), ("Virtual Care Origin", origin_tab),
          ("Caller Frequency", frequency_tab), ("Monthly Trend", trend_tab),
          ("By Group", group_tab), ("Frequent Callers", G("top_frequent_callers")),
          ("Method and Quality", method_tab)]

info = [("Run", "run identifier", RUN_ID),
        ("Run", "source file", SOURCE_FILE),
        ("Run", "calls loaded", f"{len(raw):,}"),
        ("Run", "calls after cleaning", f"{len(df):,}"),
        ("Run", "call dates", f"{df['call_dt'].min():%Y-%m-%d} to {df['call_dt'].max():%Y-%m-%d}"),
        ("Run", "calls tied to a person", f"{len(ident):,} ({len(ident)/len(df)*100:.1f}% of calls)"),
        ("Run", "people identified", f"{n_persons:,}")]
info += [("Findings", "people who called more than once",
          f"{len(multi):,} ({len(multi)/len(per)*100:.1f}% of people), accounting for {int(multi['calls'].sum()):,} calls "
          f"({multi['calls'].sum()/per['calls'].sum()*100:.1f}% of calls tied to a person)")]
for _, r in rates.iterrows():
    info.append(("Findings", f"return rate within {r['Window']}",
                 f"{r['Return rate']}% of calls were followed by another call from the same person; "
                 f"{r['Share of people with a return']}% of people had at least one return"))
info.append(("Findings", "repeat calls",
             f"{n_repeat:,} of {len(d):,} calls tied to a person, {n_repeat/len(d)*100:.1f} percent"))
info.append(("Findings", "repeat calls on the same calendar day",
             f"{n_same_day:,}, {n_same_day/n_repeat*100:.1f} percent of repeat calls"))
for k in [3, 5, 7, 30]:
    n = int((d.loc[d['is_repeat'], 'prev_gap_days'] <= k).sum())
    info.append(("Findings", f"repeat calls within {k} days", f"{n:,}, {n/n_repeat*100:.1f} percent of repeat calls"))
info.append(("Findings", "people to check before sharing a caller list", f"{len(review):,}"))

info += [
 ("How the data was derived", "step 1 - load and clean",
  "Calls are loaded from the source file. Rows without a usable call date are dropped. Rows sharing a 911 incident number are collapsed to the earliest row, so one incident is never counted as a repeat of itself."),
 ("How the data was derived", "step 2 - tidy up names, dates of birth, and phones",
  "Names are lowercased with punctuation and suffixes removed. Dates of birth are rejected if impossible. Phones are reduced to ten digits and rejected if they are not dialable."),
 ("How the data was derived", "step 3 - drop placeholders",
  "Placeholder names such as John Doe and Unknown, dates of birth shared by an implausible number of different last names, and phones used by more than five different people are left out of matching. A call needs a usable name plus either a date of birth or a phone to be tied to a person."),
 ("How the data was derived", "step 4 - group calls into people",
  "Five rules run in order: exact name and date of birth; similar first name with same last name and date of birth; similar last name with same first name and date of birth; same first name, date of birth and phone with any last name; and same name and phone where the calls share one date of birth or none. Calls are never grouped on name alone."),
 ("How the data was derived", "step 5 - how names are compared",
  "Similarity is the Jaro-Winkler score, a standard name-comparison measure from 0 to 1, with a threshold of 0.88. Smith and Smyth score 0.89. Nicknames such as Bob and Robert are matched from a lookup list."),
 ("How the data was derived", "step 6 - limit on chains",
  "A person is capped at 3 different first names and 3 different last names. Without the cap, A links to B and B links to C until many different people who share a date of birth become one person. Links the cap rejected are counted on the Method and Quality tab."),
 ("How the data was derived", "step 7 - measure returns",
  "For each call, the time to that person's next call is measured. A call counts as having a return if the next call falls within the window. Only calls with a full follow-up window before the end of the data are counted, so calls in the final 30, 60, or 90 days do not drag the rates down."),
 ("How to read it", "what a return means",
  "A return is a separate 911 incident from the same person within the window after one of their calls."),
 ("How to read it", "the rates are a floor",
  "Calls we cannot tie to a person are left out, and the chaining cap rejects some genuine links, so the true repeat rate is at or above these figures."),
 ("How to read it", "two ways to count",
  "Return rate is per call. The people columns answer the different question of how many people ever came back."),
 ("How to read it", "before sharing a caller list",
  "Check the Needs review column on the Frequent Callers tab. Those people rest on the weakest matching evidence and should be confirmed in the source data."),
 ("How to read it", "behavioral health rows",
  "The behavioral health rows on the By Group tab use the same keyword match as the behavioral health workbook. It is not checked here, so read it as a direction rather than a measured difference."),
 ("How to read it", "privacy",
  "No names, dates of birth, or phone numbers appear in this workbook. Each person is labeled with a generated key that is meaningful only within this run."),

 ("Columns - Repeat Calls", "Section", "Repeat calls in total, same calendar day or later, or the share of calls followed by a repeat."),
 ("Columns - Repeat Timing", "Section", "Cumulative counts by window, or the individual gap bands."),
 ("Columns - Repeat Timing", "Item", "The window, such as 7 days, or the gap band, such as 4-5 days. Window rows are cumulative and gap bands are not."),
 ("Columns - Virtual Care Origin", "Disposition of the previous call", "The disposition recorded on the call before a repeat call that was dispositioned virtual care."),
 ("Columns - Virtual Care Origin", "Repeat calls", "Repeat calls dispositioned virtual care whose previous call had that disposition. Covers every such repeat call."),
 ("Columns - Virtual Care Origin", "Share of these repeat calls", "That count divided by all repeat calls dispositioned virtual care."),
 ("Columns - Monthly Trend", "Month", "Calendar month of the call that was followed by a repeat."),
 ("Columns - Monthly Trend", "Calls measured and Calls with a return", "Calls in that month with a full 30-day follow-up window, and how many were followed by a repeat."),
 ("Columns - Repeat Calls", "Item", "The window, the gap band, the timing category, or the follow-up window."),
 ("Columns - Repeat Calls", "Repeat calls", "Calls that are a person's second or later call, counted in that band. Window rows are cumulative."),
 ("Columns - Repeat Calls", "Share of repeat calls", "That count divided by all repeat calls."),
 ("Columns - Repeat Calls", "Repeat calls excluding same calendar day", "The same count with repeat calls that fell on the same calendar day as the previous call removed."),
 ("Columns - Repeat Calls", "Calls measured and Return rate", "Calls with a full follow-up window before the end of the data, and the share of those followed by a repeat."),

 ("Columns - Disposition Flow", "Section", "Disposition of the repeat call, the previous call to repeat call counts, the row percentages, or the origin percentages."),
 ("Columns - Disposition Flow", "Item", "For the first section, the disposition on the repeat call. For the other sections, the disposition on the person's previous call."),
 ("Columns - Disposition Flow", "Measure", "For the flow sections, the disposition recorded on the repeat call."),
 ("Columns - Disposition Flow", "counts", "Repeat calls that moved from the row disposition to the column disposition. Limited to the eight most common dispositions on both sides."),
 ("Columns - Disposition Flow", "row percent", "Of the repeat calls whose previous call had the row disposition, the share that ended in each column disposition. Rows total 100."),
 ("Columns - Disposition Flow", "origin percent", "Of the repeat calls that ended in the column disposition, the share whose previous call had each row disposition. Columns total 100."),

 ("Columns - Return Rates", "Window", "The follow-up window: 30, 60, or 90 days."),
 ("Columns - Return Rates", "Calls measured", "Calls with a full follow-up window before the end of the data."),
 ("Columns - Return Rates", "Calls with a return", "Measured calls followed by another call from the same person inside the window."),
 ("Columns - Return Rates", "Return rate", "Calls with a return divided by Calls measured."),
 ("Columns - Return Rates", "People measured", "Distinct people among the measured calls."),
 ("Columns - Return Rates", "People with a return", "How many of those people came back at least once."),
 ("Columns - Return Rates", "Share of people with a return", "People with a return divided by People measured."),

 ("Columns - Caller Frequency", "Section", "Calls per person, or frequent callers by rolling window."),
 ("Columns - Caller Frequency", "Item", "The calls-per-person group, or a window and threshold such as 30 days, 3 or more calls."),
 ("Columns - Caller Frequency", "Measure", "People, Calls, Share of people, Share of calls, or Calls from these people."),
 ("Columns - Caller Frequency", "Value", "The number."),
 ("Columns - Caller Frequency", "how a threshold is counted", "A person meets a threshold if any rolling window of that length contains at least that many of their calls."),

 ("Columns - By Group", "Section", "Market, disposition of the first call, caller type, or behavioral health keywords."),
 ("Columns - By Group", "Item", "The value within that group, such as a single market or disposition."),
 ("Columns - By Group", "Calls measured (30 days)", "Calls in that group with a full 30 day follow-up window. The 60 and 90 day columns work the same way."),
 ("Columns - By Group", "Return rate (30 days)", "Return rate for that group over 30 days. Each group is measured against its own call volume."),

 ("Columns - Frequent Callers", "Person", "Generated person label. Meaningful only within this run, and not a patient identifier."),
 ("Columns - Frequent Callers", "Calls", "Total calls grouped under that person."),
 ("Columns - Frequent Callers", "First call and Last call", "Dates of their earliest and latest call."),
 ("Columns - Frequent Callers", "Days between first and last call", "How long their calls span."),
 ("Columns - Frequent Callers", "Calls with behavioral health keywords", "How many of their calls had a behavioral health keyword match."),
 ("Columns - Frequent Callers", "Most calls in any 30 days", "The most calls that person made inside any rolling 30 day period. The 60 and 90 day columns work the same way."),
 ("Columns - Frequent Callers", "Most common market and Most common disposition", "The market and disposition that appear most often across their calls."),
 ("Columns - Frequent Callers", "Needs review", "TRUE when the matching evidence for this person is weak enough to check by hand."),
 ("Columns - Frequent Callers", "Review reason", "Why it was flagged: more than one date of birth, at a name limit, or many different phones."),

 ("Columns - Method and Quality", "Section", "Field mapping, identity field profile, identity quality, or match rules."),
 ("Columns - Method and Quality", "Item", "The field, quality measure, or matching rule."),
 ("Columns - Method and Quality", "Source column", "The actual column in the source file used for that field."),
 ("Columns - Method and Quality", "Share filled", "Share of calls where that field has a usable value."),
 ("Columns - Method and Quality", "Distinct values", "How many different values appear in that field."),
 ("Columns - Method and Quality", "Calls on the most common value", "Calls sharing the single most common value. A high number points to a placeholder or a shared line."),
 ("Columns - Method and Quality", "Calls linked", "Calls joined to an existing person by that rule."),
 ("Columns - Method and Quality", "Links rejected by the name limit", "Links that rule proposed but the chaining cap rejected."),
]
data_info = pd.DataFrame(info, columns=["Section","Item","Detail"])

xlsx = os.path.join(OUT_DIR, f"Nurse_Navigation_Repeat_Callers_{RUN_ID}.xlsx")
try: import xlsxwriter; eng = "xlsxwriter"
except ImportError: eng = "openpyxl"
with pd.ExcelWriter(xlsx, engine=eng) as wtr:
    for tab, frame in OUTPUT:
        if tab == "Data Info":
            data_info.to_excel(wtr, sheet_name=tab, index=False)
        elif frame is not None and len(frame):
            sanitize(frame).to_excel(wtr, sheet_name=tab[:31], index=False)
        else:
            continue
        print("added", tab)
for old in glob.glob(os.path.join(OUT_DIR, "Nurse_Navigation_Repeat_Callers_*.xlsx")):
    if os.path.abspath(old) != os.path.abspath(xlsx):
        try: os.remove(old)
        except Exception: pass
print("output:", os.path.basename(xlsx))